# 02 · Experimental design: what can go wrong?
Before modelling, understand the assay.
- **CRISPRi** knocks *down* (not out) a gene; knockdown efficiency varies per guide and per gene.
- **Controls** (non-targeting guides) define the baseline. Effects are always *relative to controls from the same batch*.
- **Batches (gem groups)** are technical replicates and a source of confounding; they also provide replicates for honest statistics.
- Cells with the wrong/ambiguous guide assignment or without target suppression are not informative perturbations.
- CRISPRi (Replogle) and CRISPRa (Norman) are different experiments: never pool their expression values.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
from perturbai.data.loaders import load_dataset
adata = load_dataset(DS, P, synthetic=SYNTHETIC)
print([c for c in adata.obs.columns])
if BATCH in adata.obs:
    t = pd.crosstab(adata.obs[BATCH], adata.obs[PERT] == CTRL)
    t.columns = ["perturbed", "control"]
    t["control_frac"] = t.control / (t.perturbed + t.control)
    display(t.describe())

## Visual check of on-target knockdown (quick look; notebook 03 quantifies it)

In [ ]:
from perturbai.preprocessing.qc import normalize_log1p_matrix, choose_perturbations, subsample_cells
labels = adata.obs[PERT].astype(str).values
cand = choose_perturbations(labels, CTRL, 40, 50, seed=0, candidates=set(adata.var_names))[:6]
mask = subsample_cells(labels, CTRL, cand, 3000, 300, seed=0)
sub = adata[mask].copy()
_, Xl = normalize_log1p_matrix(sub.X)
fig, axes = plt.subplots(1, len(cand), figsize=(3 * len(cand), 3), sharey=False)
for ax, g in zip(np.atleast_1d(axes), cand):
    j = list(sub.var_names).index(g); lab = sub.obs[PERT].astype(str).values
    col = Xl[:, j].toarray().ravel() if sp.issparse(Xl) else Xl[:, j]
    ax.boxplot([col[lab == CTRL], col[lab == g]], showfliers=False)
    ax.set_xticklabels(["ctrl", g], fontsize=8); ax.set_title(g, fontsize=9)
plt.suptitle("Target-gene expression (log-norm): control vs its own perturbation"); plt.tight_layout(); plt.show()

## Write-up (for your report)
Summarise: assay principle, controls, batches, expected failure modes, and which columns identify them in this file.